# Building a Summarization Node from Scratch

In the previous notebook we gave our agent perfect memory with checkpointers. That created a new problem.

**The conversation never stops growing.** Every turn appends messages to the thread, and every turn we send the entire history to the LLM. After 50 turns:

1. Cost grows on every single call (you pay for the whole history every time)
2. Latency grows with input size
3. Eventually you hit the context window limit and the app crashes
4. LLMs get worse at following instructions when buried in irrelevant history

Production fix: when the conversation gets long, **summarize the old messages into a short running summary, delete them from state, and keep only the recent turns**. The agent keeps the knowledge but drops the bulk.

We will build this by hand, because you should know exactly what happens to your context window.

## The plan

```
new user message arrives
        |
        v
+------------------+     too many messages?
| summarize node   | --- yes: fold old messages into summary,
|                  |          delete them, keep last few
|                  | --- no: do nothing, pass through
        |
        v
+------------------+
| chat node        |  sees: summary (as system message) + recent messages
+------------------+
```

Two design decisions you can tune later:

* **Trigger**: we use message count (more than 6). Token count works too.
* **Keep window**: we keep the last 3 messages untouched so the LLM still sees the immediate back and forth verbatim.

In [1]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

## Step 1: extend the state with a summary field

`MessagesState` gives us the `messages` list with its append behavior. We add one more key, `summary`, a plain string. When a node returns `{"summary": "..."}` it simply overwrites the old value.

In [2]:
from langgraph.graph import MessagesState


class State(MessagesState):
    summary: str

## Step 2: the summarization node

This is the heart of the lesson. Read it top to bottom:

1. If the conversation is short, do nothing
2. Otherwise split messages into old ones and the recent window
3. Ask the LLM to compress the old ones, merging with any existing summary
4. Return the new summary, plus `RemoveMessage` objects that delete the old messages from state

`RemoveMessage` is the special trick: the `messages` reducer in LangGraph understands it as "delete the message with this id from state".

In [3]:
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

MAX_MESSAGES = 6   # trigger: summarize when history exceeds this
KEEP_LAST = 3      # how many recent messages survive untouched


def summarize_node(state: State):
    messages = state["messages"]

    if len(messages) <= MAX_MESSAGES:
        return {}

    old_messages = messages[:-KEEP_LAST]
    existing_summary = state.get("summary", "")

    if existing_summary:
        instruction = (
            f"This is the current summary of the conversation so far:\n"
            f"{existing_summary}\n\n"
            f"Extend the summary with the new messages above. "
            f"Keep every concrete fact: names, products, prices, decisions."
        )
    else:
        instruction = (
            "Summarize the conversation above. "
            "Keep every concrete fact: names, products, prices, decisions."
        )

    summary_response = llm.invoke(
        old_messages + [HumanMessage(content=instruction)]
    )

    delete_ops = [RemoveMessage(id=m.id) for m in old_messages]

    return {"summary": summary_response.content, "messages": delete_ops}

## Step 3: the chat node injects the summary

The LLM never sees the deleted messages again. Instead it sees one system message carrying the compressed knowledge, followed by the recent window.

In [4]:
from langchain_core.messages import SystemMessage


def chat_node(state: State):
    summary = state.get("summary", "")

    if summary:
        system = SystemMessage(
            content=f"Summary of the conversation so far:\n{summary}"
        )
        context = [system] + state["messages"]
    else:
        context = state["messages"]

    response = llm.invoke(context)
    return {"messages": [response]}

## Step 4: wire the graph

Summarization runs **before** the chat node on every turn. When the history is short it is a no-op that costs nothing.

In [5]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(State)
builder.add_node("summarize", summarize_node)
builder.add_node("chat", chat_node)
builder.add_edge(START, "summarize")
builder.add_edge("summarize", "chat")
builder.add_edge("chat", END)

graph = builder.compile(checkpointer=InMemorySaver())

## Step 5: test with a growing conversation

We feed the agent a realistic seller conversation. The very first message contains facts (name, company, budget) that we will ask about at the end, after those messages have been summarized away.

In [6]:
turns = [
    "Hi, I am Satvik from BoomAudio. We sell wireless earbuds and my monthly ad budget is 50000 rupees.",
    "Our main competitor is boAt Airdopes, they sell at 1299 rupees.",
    "We are pricing ours at 1499 rupees. Is that too high against boAt?",
    "What discount strategy would you suggest for Diwali season?",
    "Should I run Amazon ads or Instagram ads with my budget?",
]

config = {"configurable": {"thread_id": "seller-chat"}}

for turn in turns:
    result = graph.invoke({"messages": [("user", turn)]}, config)
    state = graph.get_state(config)
    n = len(state.values["messages"])
    has_summary = bool(state.values.get("summary"))
    print(f"messages in state: {n:2d} | summary exists: {has_summary} | user: {turn[:55]}")

messages in state:  2 | summary exists: False | user: Hi, I am Satvik from BoomAudio. We sell wireless earbud


messages in state:  4 | summary exists: False | user: Our main competitor is boAt Airdopes, they sell at 1299


messages in state:  6 | summary exists: False | user: We are pricing ours at 1499 rupees. Is that too high ag


messages in state:  4 | summary exists: True | user: What discount strategy would you suggest for Diwali sea


messages in state:  6 | summary exists: True | user: Should I run Amazon ads or Instagram ads with my budget


Watch the message count: it grows 2 per turn until it crosses the threshold, then collapses. Let us look at what the summarizer wrote.

In [7]:
state = graph.get_state(config)

print("MESSAGES NOW IN STATE:")
for m in state.values["messages"]:
    print(f"  [{m.type}] {m.content[:70]}")

print()
print("RUNNING SUMMARY:")
print(state.values["summary"])

MESSAGES NOW IN STATE:
  [human] We are pricing ours at 1499 rupees. Is that too high against boAt?
  [ai] Pricing your wireless earbuds at 1,499 rupees, while slightly higher t
  [human] What discount strategy would you suggest for Diwali season?
  [ai] For the Diwali season, implementing a well-thought-out discount strate
  [human] Should I run Amazon ads or Instagram ads with my budget?
  [ai] Deciding between Amazon ads and Instagram ads depends on your target a

RUNNING SUMMARY:
**Summary of Conversation:**

- **User**: Satvik from BoomAudio
- **Product**: Wireless earbuds
- **Monthly Ad Budget**: 50,000 rupees
- **Main Competitor**: boAt Airdopes
- **Competitor Price**: 1,299 rupees

**Marketing Strategies Discussed**:
1. **Highlight Unique Features**: Emphasize superior sound quality, battery life, ergonomic design, etc.
2. **Competitive Pricing Strategy**: Consider pricing at or slightly below 1,299 rupees and offer bundling options.
3. **Customer Testimonials and Reviews**: En

## Step 6: the real test

The facts from turn 1 (name, company, budget) only exist inside the summary now. The original messages are deleted. If the agent answers correctly, our node works.

In [8]:
result = graph.invoke(
    {"messages": [("user", "Remind me: what is my name, my company, and my exact ad budget?")]},
    config,
)
print(result["messages"][-1].content)

Your name is Satvik, you are from BoomAudio, and your exact ad budget is 50,000 rupees per month.


## Measuring what we saved

A rough comparison of what we send to the LLM with and without summarization, using token counts.

In [9]:
# The deleted messages still exist in older checkpoints, so we can
# reconstruct what the context WOULD have been without summarization.
history = list(graph.get_state_history(config))
max_messages_ever = max(len(s.values.get("messages", [])) for s in history)

state = graph.get_state(config)
kept = state.values["messages"]
kept_chars = sum(len(m.content) for m in kept) + len(state.values["summary"])

# every checkpoint stores the messages it had at that step; the longest one
# before any deletion shows the raw accumulated history
raw_chars = 0
for snap in history:
    chars = sum(len(m.content) for m in snap.values.get("messages", []))
    raw_chars = max(raw_chars, chars + len(snap.values.get("summary", "") or ""))

print(f"context sent to the LLM without summarization would grow every turn")
print(f"largest raw history seen in checkpoints: {raw_chars} chars")
print(f"context now, after summarization:        {kept_chars} chars")
print(f"messages in state: {len(kept)} instead of {max_messages_ever} and climbing")

context sent to the LLM without summarization would grow every turn
largest raw history seen in checkpoints: 11746 chars
context now, after summarization:        5793 chars
messages in state: 4 instead of 7 and climbing


## Key takeaways

1. Checkpointers remember everything, which makes context grow without bound
2. A summarization node compresses old messages into a running summary and deletes them with `RemoveMessage`
3. The summary is injected as a system message, so no knowledge is lost, only detail
4. Trigger threshold and keep window are tuning knobs: message count is simple, token count is more precise
5. The prebuilt `SummarizationNode` in the `langmem` package does the same thing. Now that you have built it by hand, you are allowed to use the prebuilt one at work

In the MarketPulse backend (`backend/graph.py`) this exact node runs in front of a tool calling agent, with one extra subtlety: we never split in the middle of a tool call sequence. You will see that in the backend walkthrough.